In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
# walks upward until it finds a directory containing ppm/
PROJECT_ROOT = next(
    (p for p in (current_dir, *current_dir.parents) if (p / "ppm").is_dir()),
    current_dir,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from ppm.wandb_utils import load_multiple_experiments

plots_dir = PROJECT_ROOT / "plots/features"
plots_dir.mkdir(parents=True, exist_ok=True)

In [2]:
PROJECTS=[
  "BPI12_spark_freeze_001",
  "BPI12_spark_freeze_001_tied",
]

runs_raw, _ = load_multiple_experiments(PROJECTS, force_update=False)


Database already exists: /app/visualization/paper/metrics/BPI12_spark_freeze_001.db
Use force_update=True to re-fetch from wandb
Fetched 10 runs and 207 history entries to /app/visualization/paper/metrics/BPI12_spark_freeze_001_tied.db
Loaded 18 runs from 2 experiments
Datasets: ['BPI12']


In [3]:
import pandas as pd

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", None)         # don't wrap/truncate by width
pd.set_option("display.max_colwidth", None)  # optional: full cell content

runs_raw.columns

Index(['id', 'name', 'r', 'lr', 'log', 'device', 'epochs', 'compile',
       'n_heads', 'backbone', 'n_layers', 'patience', 'rnn_type', 'strategy',
       'val_size', 'grad_clip', 'lifecycle', 'min_delta', 'precision',
       'val_split', 'batch_size', 'lora_alpha', 'fine_tuning', 'hidden_size',
       'num_workers', 'total_params', 'weight_decay', 'freeze_layers',
       'embedding_size', 'trainable_params', 'continuous_targets',
       'categorical_targets', 'continuous_features', 'categorical_features',
       'memory_safety_margin', 'time_positional_encoding', 'duration_sec',
       'best_test_final_loss', 'best_test_final_next_activity_acc',
       'best_test_final_next_activity_f1',
       'best_test_final_next_activity_loss', 'dataset/filtered_dataset_cases',
       'dataset/filtered_dataset_end', 'dataset/filtered_dataset_events',
       'dataset/filtered_dataset_max_duration_days',
       'dataset/filtered_dataset_start', 'dataset/orig_dataset_cases',
       'dataset/orig_data

In [7]:
# prepocessing
runs = runs_raw.copy()

if "weight_tying" in runs.columns:
    # Normalize booleans then force untied for base project / missing values.
    runs["weight_tying"] = runs["weight_tying"].replace({"True": True, "False": False})
    mask_base_project = runs["project"].eq("BPI12_spark_freeze_001")
    runs.loc[mask_base_project | runs["weight_tying"].isna(), "weight_tying"] = False

fine_tuning_col = "fine_tuning" if "fine_tuning" in runs.columns else "fine-tuning"
freeze_layers_col = "freeze_layers" if "freeze_layers" in runs.columns else "freeze-layers"
mask_no_finetuning = runs[fine_tuning_col].isna() | (runs[fine_tuning_col].astype(str) == "None")
runs.loc[mask_no_finetuning, freeze_layers_col] = "all"


## Impact tying

In [9]:
#METRIC = "best_test_final_next_activity_acc"
METRIC = "best_test_final_next_activity_f1"

for col in ["categorical_features", "continuous_features"]:
    if col in runs.columns:
        runs[col] = runs[col].astype(str)

GROUP_COLS = ["log", "backbone", "weight_tying", "freeze_layers", "lr", "batch_size", "categorical_features", "continuous_features"]

EXTRA_COLS = ["project", "total_params", "trainable_params", "embedding_size"]
EXTRA_COLS = [c for c in EXTRA_COLS if c in runs.columns]  # only keep ones that exist

# Build df with everything we need
df = runs[GROUP_COLS + ["id", METRIC] + EXTRA_COLS].dropna(subset=[METRIC]).copy()
df[METRIC] = df[METRIC].astype(float)

# 1) Metric stats (ONLY from METRIC)
agg = (
    df.groupby(GROUP_COLS)[METRIC]
      .agg(["count", "mean", "std", "min", "max"])
      .rename(columns={
          "count": "n_runs",
          "mean": "acc_mean",
          "std": "acc_std",
          "min": "acc_min",
          "max": "acc_max",
      })
)

# 2) Add “descriptor” columns (not metric-related)
#    We assume these are constant within group; if not, we’ll detect it below.
descriptors = df.groupby(GROUP_COLS)[EXTRA_COLS].first() if EXTRA_COLS else None

# Optional: detect groups where descriptors are NOT constant
if EXTRA_COLS:
    nunique = df.groupby(GROUP_COLS)[EXTRA_COLS].nunique(dropna=False)
    bad = (nunique > 1).any(axis=1)
    if bad.any():
        print("Warning: some groups have varying descriptor values:")
        display(nunique[bad].reset_index())

# 3) Best run id per group (based on METRIC)
best_idx = df.groupby(GROUP_COLS)[METRIC].idxmax()
best_runs = df.loc[best_idx].set_index(GROUP_COLS)["id"].rename("best_run_id")

# 4) Combine
summary = agg
if descriptors is not None:
    summary = summary.join(descriptors)

summary = summary.join(best_runs).reset_index()
summary["n_runs"] = summary["n_runs"].astype(int)

for col in ["acc_mean", "acc_std", "acc_min", "acc_max"]:
    summary[col] = summary[col].map(lambda x: f"{x:.4f}")

summary = summary.sort_values(GROUP_COLS).reset_index(drop=True)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
    display(summary)


,log,backbone,weight_tying,freeze_layers,lr,batch_size,categorical_features,continuous_features,n_runs,acc_mean,acc_std,acc_min,acc_max,project,total_params,trainable_params,embedding_size,best_run_id
0,BPI12,gpt2,False,"1,-1",0.0050,64,"['activity', 'resource']","['accumulated_time', 'amount']",8,0.7166,0.0092,0.7074,0.7320,BPI12_spark_freeze_001,124552743,14288679,768,wnlzc31l
1,BPI12,gpt2,True,"1,-1",0.0005,64,"['activity', 'resource']","['accumulated_time', 'amount']",5,0.6989,0.0061,0.6921,0.7036,BPI12_spark_freeze_001_tied,124522791,14258727,768,8b9ak4s5
2,BPI12,gpt2,True,"1,-1",0.0050,64,"['activity', 'resource']","['accumulated_time', 'amount']",5,0.7099,0.0124,0.6945,0.7289,BPI12_spark_freeze_001_tied,124522791,14258727,768,jc1f3ukt


In [6]:
runs.weight_tying.unique()

array([nan, True], dtype=object)